### IMPORTAÇÕES

In [1]:
# IMPORTANDO BIBLIOTECAS
import pandas as pd
import awswrangler as awr
import openpyxl
import shutil
import datetime as dt
import pyautogui
import time
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  

In [2]:
# FORMATANDO EM CAIXA ALTA
def format_type(df):
    for col in df.select_dtypes(include=['string']).columns:
        df[col] = df[col].str.upper()
    return df

### EXTRAÇÕES DE BASES

In [3]:
# EXTRAINDO DADOS DE ATIVOS

query_ativos = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_ATIVOS.sql"
with open(query_ativos, 'r', encoding='utf-8') as file:
    sql_ativos = file.read()   

df_ativos =awr.athena.read_sql_query(
    sql=sql_ativos,database='silver'
)



In [4]:
# ELIMINANDO DUPLICATAS DE ATIVOS POR CHASSI

df_ativos = df_ativos.drop(columns=['rn'])

df_ativos = (
    df_ativos
    .drop_duplicates(subset=['chassi'], keep='first')
)

df_ativos = df_ativos.sort_values(
    by=['inicio_vig', 'data_ativacao'], ascending=False
).reset_index(drop=True)

In [5]:
# EXTRAINDO DADOS DE CANCELAMENTOS INTEGRAIS (CONJUNTO)

query_cancelados_integrais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_INTEGRAIS.sql"
with open(query_cancelados_integrais, 'r', encoding='utf-8') as file:
    sql_cancelados_integrais = file.read()   

df_cancelamentos_integrais =awr.athena.read_sql_query(
    sql=sql_cancelados_integrais, database='silver'
)

In [6]:
# EXTRAINDO DADOS DE CANCELAMENTOS PARCIAIS (CONJUNTO)

query_cancelados_parciais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_PARCIAIS.sql"
with open(query_cancelados_parciais, 'r', encoding='utf-8') as file:
    sql_cancelados_parciais = file.read()   

df_cancelamentos_parciais =awr.athena.read_sql_query(
    sql=sql_cancelados_parciais,database='silver'
)

In [7]:
# FORMATAÇÃO EM CAIXA ALTA
format_type(df_ativos)
format_type(df_cancelamentos_integrais)
format_type(df_cancelamentos_parciais)

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,usuario_cancelamento,coverage_id,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_atualizacao,empresa
0,OCX2777,93ZS2MRH0B8813421,19089,19089,<NA>,9704,13584,UNIDADE CASCAVEL,MAURICIO BELLO,ATIVO,...,,84544,VIDROS,CANCELADO,2026-02-19,2025-02-12,2025-02-14,2025-02-14,2026-02-05,STCOOP
1,OCX2777,93ZS2MRH0B8813421,19089,19089,<NA>,9704,13584,UNIDADE CASCAVEL,MAURICIO BELLO,ATIVO,...,,84542,CASCO (VEÍCULO),CANCELADO,2026-02-19,2025-02-12,2025-02-14,2025-02-14,2026-02-05,STCOOP
2,DJB2996,93KK6AAC44E100045,12208,12208,<NA>,11915,16420,MF - MARTUCCI SOLUCOES,AMILTON MARTUCCI,ATIVO,...,,101525,VIDROS,CANCELADO,2026-02-19,2025-11-21,2025-12-29,2025-12-29,2026-01-30,STCOOP
3,DJB2996,93KK6AAC44E100045,12208,12208,<NA>,11915,16420,MF - MARTUCCI SOLUCOES,AMILTON MARTUCCI,ATIVO,...,,101521,CASCO (COMPLEMENTO),CANCELADO,2026-02-19,2025-11-21,2025-12-29,2025-12-29,2026-01-30,STCOOP
4,QHL2833,9AA09123GFC134018,16393,0,16393,3518,14372,UNIDADE ITAJAI,JOSÉ ELIAS DOS SANTOS FILHO,ATIVO,...,,89310,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-02-19,2025-05-07,2025-05-09,2025-05-09,2025-10-03,STCOOP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2352,AAW5718,9EL11FR031V005774,69423,0,69423,12315,140280,UNIDADE JOINVILLE,ANA BEATRIZ PRIOSTE,ATIVO,...,,675904,REPARAÇÃO OU REPOSIÇÃO DO (SEMI)REBOQUE,CANCELADO,2026-02-19,2025-04-10,2025-04-19,2025-04-19,2025-10-24,SEGTRUCK
2353,FRF4348,93ZM2SSH0E8826134,60020,60020,<NA>,23984,140570,UNIDADE MARINGÁ,ICARO GOMES,ATIVO,...,,677634,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-02-19,2025-11-11,2025-11-17,2025-11-17,2026-01-22,SEGTRUCK
2354,FRF4348,93ZM2SSH0E8826134,60020,60020,<NA>,23984,140570,UNIDADE MARINGÁ,ICARO GOMES,ATIVO,...,,677632,REPARAÇÃO OU REPOSIÇÃO DO VEÍCULO,CANCELADO,2026-02-19,2025-11-11,2025-11-17,2025-11-17,2026-01-22,SEGTRUCK
2355,MIE8B71,9534N8248BR121127,122939,122939,<NA>,44092,140228,UNIDADE MARINGÁ,<NA>,ATIVO,...,,675554,REPARAÇÃO OU REPOSIÇÃO DO COMPLEMENTO,CANCELADO,2026-02-19,2025-03-20,2025-03-22,2025-03-22,2026-02-04,SEGTRUCK


In [8]:
# DEFININDO TODAY TS e YESTERDAY

today_ts = pd.Timestamp.today().normalize()

if today_ts.weekday() == 0:  
    yesterday_ts = today_ts - dt.timedelta(days=3)
else:
    yesterday_ts = today_ts - dt.timedelta(days=1)
yesterday = yesterday_ts.strftime('%d-%m-%Y')

yesterday

'18-02-2026'

### JUNÇÃO DE BASES DE CANCELAMENTO

In [9]:
# FORMATANDO COLUNAS DE DATA DE CANCELAMENTO PARA DT (TS)

df_cancelamentos_integrais['data_cancelamento'] = pd.to_datetime(
	df_cancelamentos_integrais['data_cancelamento'], errors='coerce'
).dt.date

placas_canceladas_dia_anterior = df_cancelamentos_integrais.loc[
	df_cancelamentos_integrais['data_cancelamento'] == yesterday_ts.date(),
	'placa'
].unique().tolist()

In [10]:
# CRIANDO COLUNAS DE IDENTIFICADOR DE CANCELAMENTO

df_cancelamentos_integrais['identificador'] = 'INTEGRAL'
df_cancelamentos_parciais['identificador'] = 'PARCIAL'

In [11]:
# RETIRANDO DE CANCELAMENTOS PARCIAIS AS PLACAS AINDA ATIVAS

lista_ativos = df_ativos['chassi'].to_list()
df_cancelamentos_parciais = df_cancelamentos_parciais[~df_cancelamentos_parciais['chassi'].isin(lista_ativos)]

In [12]:
# CONCATENANDO BASES DE CANCELAMENTO E RETIRANDO DUPLICATAS POR CHASSI

df_cancelamentos = pd.concat(
    [df_cancelamentos_integrais, df_cancelamentos_parciais], ignore_index=True
)

df_cancelamentos = df_cancelamentos.sort_values(
    by='data_cancelamento', ascending=False
).reset_index(drop=True)

df_cancelamentos.drop_duplicates(subset=['chassi'], keep='first')

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_cancelamento,empresa,identificador,data_atualizacao
0,QJR1B87,9BVRG20C1KE861261,15239818,15239818,<NA>,4242,8939,OPENTRUCK CORRETORA,PAULO FERNANDES,FINALIZADO,...,REPARAÇÃO A TERCEIROS,ATIVO,2026-02-19,2025-01-27,2025-02-03,2025-02-03,2026-02-19,VIAVANTE,INTEGRAL,NaT
1,BEJ8E14,98PTT47MSLB111956,15931234,15931234,<NA>,6029,9296,UNIDADE MARINGÁ,PAULO HENRIQUE DE SOUZA LOCATELI,FINALIZADO,...,CASCO (VEÍCULO),ATIVO,2026-02-19,2025-01-31,2025-02-03,2025-02-03,2026-02-19,VIAVANTE,INTEGRAL,NaT
2,AEE5909,9BSR6X400B3678177,15926914,15926914,<NA>,5927,9075,UNIDADE LONDRINA,DAIANE CRISTINA VEIGA DA SILVA,FINALIZADO,...,CASCO (VEÍCULO),ATIVO,2026-02-19,2025-01-29,2025-02-03,2025-02-03,2026-02-19,VIAVANTE,INTEGRAL,NaT
3,SFC5B08,91VB0952PRC210859,15366,0,15366,5275,9165,UNIDADE MARINGÁ,JOÃO BATISTA VOLPATO,FINALIZADO,...,CASCO (R/SR),ATIVO,2026-02-19,2025-01-30,2025-02-03,2025-02-03,2026-02-19,VIAVANTE,INTEGRAL,NaT
4,SFM1D03,9BSR6X400R4070470,13446724,13446724,<NA>,3731,9324,UNIDADE PONTA GROSSA,LUANA SOARES ALVES DOS REIS,FINALIZADO,...,REPARAÇÃO A TERCEIROS,ATIVO,2026-02-19,2025-01-31,2025-02-03,2025-02-03,2026-02-19,VIAVANTE,INTEGRAL,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39831,BBB6F74,9EP071230H1000181,29934,0,29934,3777,21282,MICRO A - DEUZIMAR DE OLIVEIRA SENRA,MICRO A - DEUZIMAR,ATIVO,...,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-02-19,2025-07-25,2025-07-25,2025-07-27,NaN,VIAVANTE,PARCIAL,2025-11-21
39834,IRJ7D27,9BVAS02D5BE766580,16277144,16277144,<NA>,7533,12341,UNIDADE CUIABA,ROSILDA MARTINS DA CRUZ,ATIVO,...,REPARAÇÃO A TERCEIROS,CANCELADO,2026-02-19,2025-03-18,2025-03-18,2025-03-18,NaN,VIAVANTE,PARCIAL,2025-12-05
39851,SSZ2E98,91VG1354RRC214026,20811,0,20811,7774,12964,YOU SAFER C,VICTORIA CHANG DE MORAIS,ATIVO,...,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-02-19,2025-03-26,2025-04-03,2025-04-03,NaN,VIAVANTE,PARCIAL,2026-02-13
39864,AAW5718,9EL11FR031V005774,69423,0,69423,12315,140280,UNIDADE JOINVILLE,ANA BEATRIZ PRIOSTE,ATIVO,...,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-02-19,2025-04-10,2025-04-19,2025-04-19,NaN,SEGTRUCK,PARCIAL,2025-10-24


In [13]:
# DEFININDO TODAY e YESTERDAY

today = pd.Timestamp.today().date()

yesterday = today - dt.timedelta(days=1)
sexta = yesterday - dt.timedelta(days=2)
sabado = yesterday - dt.timedelta(days=1)

### CONSTRUINDO EXCEL

In [14]:
# DEFININDO CÉLULAS RELATÓRIO w4
df_cancelamentos['data_cancelamento'] = pd.to_datetime(df_cancelamentos['data_cancelamento'], errors='coerce').dt.date

if today.weekday() == 0: 
    ativos_mask = (df_ativos['inicio_vig'] >= sexta) & (df_ativos['inicio_vig'] <= yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] >= sexta) & (df_cancelamentos['data_cancelamento'] <= yesterday)
else:
    ativos_mask = (df_ativos['inicio_vig'] == yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] == yesterday)

ativados_seg = len(df_ativos[(df_ativos['empresa'] == 'SEGTRUCK') & ativos_mask])
ativados_st = len(df_ativos[(df_ativos['empresa'] == 'STCOOP') & ativos_mask])
ativados_viav = len(df_ativos[(df_ativos['empresa'] == 'VIAVANTE') & ativos_mask])
ativados_tag = len(df_ativos[(df_ativos['empresa'] == 'TAG') & ativos_mask])
total_ativados = len(df_ativos[ativos_mask])


cancelados_seg = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'SEGTRUCK') & cancelamentos_mask])
cancelados_st = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'STCOOP') & cancelamentos_mask])
cancelados_viav = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'VIAVANTE') & cancelamentos_mask])
cancelados_tag = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'TAG') & cancelamentos_mask])
total_cancelados = len(df_cancelamentos[cancelamentos_mask])

In [15]:
# FUNÇÃO PARA LIMPAR DADOS DA PLANILHA
def clear_sheet(sheet):
    max_row = sheet.max_row
    if max_row > 1:
        sheet.delete_rows(1,max_row)

In [16]:
# DEFININDO NOMES DAS ABAS E FAZENDO A LIMPEZA DE w1 e w2

template = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\template\TEMPLATE_BASE_ATIVACOES_CANCELAMENTOS.xlsx"

wb = openpyxl.load_workbook(template)

w1 = wb['ATIVOS']
w2 = wb['CANCELAMENTOS']
w3 = wb['BASE']
w4 = wb['RELATORIO']

clear_sheet(w1)
clear_sheet(w2)

In [17]:
# FORMATANDO LINHAS NULAS

numeric_cols = df_ativos.select_dtypes(include=['number']).columns
string_cols = df_ativos.select_dtypes(include=['object', 'string']).columns

df_ativos[numeric_cols] = df_ativos[numeric_cols].fillna(0)
df_ativos[string_cols] = df_ativos[string_cols].fillna('N/A')

numeric_cols = df_cancelamentos.select_dtypes(include=['number']).columns
string_cols = df_cancelamentos.select_dtypes(include=['object', 'string']).columns

df_cancelamentos[numeric_cols] = df_cancelamentos[numeric_cols].fillna(0)
df_cancelamentos[string_cols] = df_cancelamentos[string_cols].fillna('N/A')

W1 - ATIVOS

In [18]:

for c_idx, col_name in enumerate(df_ativos.columns, start=1):
    w1.cell(row=1, column=c_idx, value=col_name)

if not df_ativos.empty:
    for r_idx, row in enumerate(df_ativos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w1.cell(row=r_idx, column=c_idx, value=value)

W2 - CANCELADOS

In [19]:

for c_idx, col_name in enumerate(df_cancelamentos.columns, start=1):
    w2.cell(row=1, column=c_idx, value=col_name)

if  df_cancelamentos.empty == False:
    for r_idx, row in enumerate(df_cancelamentos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w2.cell(row=r_idx, column=c_idx, value=value)

W3 - BASE

In [20]:
first_empty_row = 1
for row in range(1, w3.max_row + 1):
    if w3.cell(row=row, column=2).value is None:
        first_empty_row = row
        break
else:
    first_empty_row = w3.max_row + 1

# PEGAR DATA ATUAL NA PLANILHA
data_atual_planilha = w3['A' + str(first_empty_row)].value.date()


In [21]:
# VERIFICA SE DATA NA PLANILHA = DATA DE ONTEM  
#if data_atual_planilha == yesterday:
    # Filtra ativos até a data de ontem (inclusive)
if "inicio_vig" in df_ativos.columns:
    # Supondo que data_ativacao já seja datetime ou string convertível
    data_ativacao_col = pd.to_datetime(df_ativos["inicio_vig"], errors='coerce')
    df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday)]
    qtd_ativos = df_ativos_filtrados['chassi'].nunique()
else:
    qtd_ativos = df_ativos['chassi'].nunique()  # fallback: considera todos

#definindo ativados totais

data_ativacao_col = pd.to_datetime(df_ativos["inicio_vig"], errors='coerce')

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday)]
qtd_ativos_dom = df_ativos_filtrados['chassi'].nunique()

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(sabado)]
qtd_ativos_sab = df_ativos_filtrados['chassi'].nunique()

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(sexta)]
qtd_ativos_sex = df_ativos_filtrados['chassi'].nunique()


#definindo ativações de sexta e sábado
ativos_mask_dom = df_ativos['inicio_vig'] == yesterday
total_ativados_dom = len(df_ativos[ativos_mask_dom])

ativos_mask_sab = df_ativos['inicio_vig'] == sabado
total_ativados_sab = len(df_ativos[ativos_mask_sab])

ativos_mask_sex = df_ativos['inicio_vig'] == sexta
total_ativados_sex= len(df_ativos[ativos_mask_sex])

#definindo cancelamentos de sexta e sábado
cancel_mask_dom = df_cancelamentos['data_cancelamento'] == yesterday
total_cancel_dom = len(df_cancelamentos[cancel_mask_dom])

cancel_mask_sab = df_cancelamentos['data_cancelamento'] == sabado
total_cancel_sab = len(df_cancelamentos[cancel_mask_sab])

cancel_mask_sex = df_cancelamentos['data_cancelamento'] == sexta
total_cancel_sex = len(df_cancelamentos[cancel_mask_sex])

#hoje = dt.date.today()
if today.weekday() == 0 and first_empty_row >= 3:
        w3['B' + str(first_empty_row + 2)] = qtd_ativos_dom
        w3['C' + str(first_empty_row + 2)] = total_ativados_dom
        w3['D' + str(first_empty_row + 2)] = total_cancel_dom

        w3['B' + str(first_empty_row + 1)] = qtd_ativos_sab
        w3['C' + str(first_empty_row + 1)] = total_ativados_sab
        w3['D' + str(first_empty_row + 1)] = total_cancel_sab

        w3['B' + str(first_empty_row)] = qtd_ativos_sex
        w3['C' + str(first_empty_row)] = total_ativados_sex
        w3['D' + str(first_empty_row)] = total_cancel_sex

        print('Registro de ativos preenchido para domingo, sábado e ativos na aba BASE!')
else:
        w3['B' + str(first_empty_row)] = qtd_ativos
        w3['C' + str(first_empty_row)] = total_ativados
        w3['D' + str(first_empty_row)] = total_cancelados
        



W4 - RESUMO

In [22]:
import datetime 
# Garante que yesterday esteja normalizado
if isinstance(yesterday, pd.Timestamp):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.datetime):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.date):
    yest_date = yesterday
else:
    yest_date = pd.to_datetime(yesterday).date()

# Hoje
hoje = datetime.date.today()
dia_semana = hoje.weekday() 

if dia_semana == 0:  # Segunda-feira
    # calcula sexta (3 dias atrás) e domingo (ontem)
    sexta = yest_date - datetime.timedelta(days=2)
    domingo = yest_date
    sexta_str = sexta.strftime('%d/%m/%Y')
    domingo_str = domingo.strftime('%d/%m/%Y')
    resumo_periodo = f"{sexta_str} (sexta) - {domingo_str} (domingo)"
    w4['C2'] = resumo_periodo
else:
    w4['C2'] = yest_date.strftime('%d/%m/%Y')

w4['C3'] = qtd_ativos

w4['C6'] = ativados_seg
w4['C7'] = ativados_st
w4['C8'] = ativados_viav
w4['C9'] = ativados_tag

w4['D6'] = cancelados_seg
w4['D7'] = cancelados_st
w4['D8'] = cancelados_viav
w4['D9'] = cancelados_tag


### quantidade de chassis por unidade

In [23]:
# Obtém o dia da semana referente a 'yesterday'
if hasattr(yesterday, "weekday"):
    dia_semana = yesterday.weekday()
else:
    try:
        dia_semana = datetime.datetime.strptime(str(yesterday), "%Y-%m-%d").weekday()
    except Exception:
        dia_semana = None  # fallback

if dia_semana == 0:  # Segunda-feira
    # sexta = ontem - 2 dias
    sexta = yesterday - datetime.timedelta(days=2)
    # Ativos de sexta, sábado e domingo (considerando 'inicio_vig')
    period_mask = df_ativos["inicio_vig"].isin([sexta, sexta + datetime.timedelta(days=1), yesterday])
    df_ativos_periodo = df_ativos[period_mask]
    df_ativos_ontem_unidades = df_ativos_periodo.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)
    print("Ativos por unidade (sexta a domingo):")
    print(df_ativos_ontem_unidades)
    
    # Cancelamentos de sexta, sábado e domingo (considerando 'data_cancelamento')
    period_mask_cancel = df_cancelamentos["data_cancelamento"].isin([sexta, sexta + datetime.timedelta(days=1), yesterday])
    df_cancelamentos_periodo = df_cancelamentos[period_mask_cancel]
    df_cancelamentos_ontem_unidades = df_cancelamentos_periodo.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)
    print("Cancelamentos por unidade (sexta a domingo):")
    print(df_cancelamentos_ontem_unidades)

else:
    # Comportamento padrão: apenas o dia anterior
    df_ativos_ontem = df_ativos[df_ativos['inicio_vig'] == yesterday]
    df_ativos_ontem_unidades = df_ativos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)
    print("Ativos por unidade (ontem):")
    print(df_ativos_ontem_unidades)

    df_cancelamentos_ontem = df_cancelamentos[df_cancelamentos['data_cancelamento'] == yesterday]
    df_cancelamentos_ontem_unidades = df_cancelamentos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)
    print("Cancelamentos por unidade (ontem):")
    print(df_cancelamentos_ontem_unidades)



Ativos por unidade (ontem):
unidade
UNIDADE PORTO VELHO                                             9
UNIDADE CASCAVEL                                                4
UNIDADE VILHENA                                                 4
MICRO A - R A SERVICOS & CONSULTORIA LTDA                       3
TRANSDESK DIGITAL LTDA                                          2
P.I - JOSÉ ELIAS                                                2
UNIDADE LONDRINA                                                2
UNIDADE ITAPEJARA                                               2
FRANQUEADO - J BATISTA VOLPATO REPRESENTACAO COMERCIAL E PRE    1
MICRO A - JOSÉ ELIAS DOS SANTOS FILHO                           1
MICRO A - MARTUCCI SOLUCOES LTDA                                1
MICRO A - NICHOLAS MARTINS DELGADO                              1
UNIDADE ITAJAI                                                  1
UNIDADE SINOP                                                   1
Name: chassi, dtype: int64
Cancelamentos

### criando imagem das unidades

In [24]:
import os
# Garante colunas date
if not pd.api.types.is_datetime64_any_dtype(df_ativos['inicio_vig']):
    df_ativos['inicio_vig'] = pd.to_datetime(df_ativos['inicio_vig'], errors='coerce').dt.date
if not pd.api.types.is_datetime64_any_dtype(df_cancelamentos['data_cancelamento']):
    df_cancelamentos['data_cancelamento'] = pd.to_datetime(df_cancelamentos['data_cancelamento'], errors='coerce').dt.date

# Se hoje for segunda-feira, usar intervalo sexta, sábado, domingo. Caso contrário, apenas ontem
if hasattr(yesterday, "weekday"):
    dia_semana = yesterday.weekday()
else:
    try:
        dia_semana = datetime.datetime.strptime(str(yesterday), "%Y-%m-%d").weekday()
    except Exception:
        dia_semana = None  # fallback

if today.weekday() == 0:  # Segunda-feira, pegar sexta, sábado e domingo
    sexta = yesterday - datetime.timedelta(days=2)
    sabado = yesterday - datetime.timedelta(days=1)
    domingo = yesterday
    dias_periodo = [sexta, sabado, domingo]

    # Filtro para o range de sexta a domingo
    df_ativos_periodo = df_ativos[df_ativos['inicio_vig'].isin(dias_periodo)]
    df_ativos_unidades = df_ativos_periodo.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

    df_cancelamentos_periodo = df_cancelamentos[df_cancelamentos['data_cancelamento'].isin(dias_periodo)]
    df_cancelamentos_unidades = df_cancelamentos_periodo.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

    periodo_str = f'{sexta.strftime("%d/%m/%Y")} a {domingo.strftime("%d/%m/%Y")}'
    
    top_ativos_unidades = df_ativos_unidades.head(20)
    top_cancelamentos_unidades = df_cancelamentos_unidades.head(20)
else:
    # Apenas ontem
    df_ativos_ontem = df_ativos[df_ativos['inicio_vig'] == yesterday]
    df_ativos_ontem_unidades = df_ativos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

    df_cancelamentos_ontem = df_cancelamentos[df_cancelamentos['data_cancelamento'] == yesterday]
    df_cancelamentos_ontem_unidades = df_cancelamentos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

    periodo_str = yesterday.strftime('%d/%m/%Y')
    top_ativos_unidades = df_ativos_ontem_unidades.head(20)
    top_cancelamentos_unidades = df_cancelamentos_ontem_unidades.head(20)

# Criar figura com dois subplots lado a lado, ativos top 20 e cancelados top 20
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Gráfico 1: Ativos por Unidade (Top 20)
if not top_ativos_unidades.empty:
    top_ativos_unidades.plot(kind='barh', ax=ax1, color='#2ecc71', edgecolor='black', linewidth=0.5)
    ax1.set_title(f'Ativos por Unidade (Top 20)\n{periodo_str}', fontsize=14, fontweight='bold', pad=15)
    ax1.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax1.set_ylabel('', fontsize=11, fontweight='bold')
    ax1.set_xticks([])
    ax1.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
    ax1.invert_yaxis()
    xmax1 = top_ativos_unidades.values.max() if len(top_ativos_unidades) > 0 else 1
    ax1.set_xlim(0, xmax1 + max(3, int(0.15 * xmax1)))
    for i, v in enumerate(top_ativos_unidades.values):
        ax1.text(v + 0.7, i, str(int(v)), va='center', fontweight='bold')
else:
    ax1.text(0.5, 0.5, 'Sem dados disponíveis', ha='center', va='center', transform=ax1.transAxes, fontsize=12)
    ax1.set_title(f'Ativos por Unidade (Top 20)\n{periodo_str}', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax1.set_ylabel('', fontsize=11, fontweight='bold')
    ax1.set_xticks([])


# Gráfico 2: Cancelados por Unidade (Top 20)
if not top_cancelamentos_unidades.empty:
    top_cancelamentos_unidades.plot(kind='barh', ax=ax2, color='#e74c3c', edgecolor='black', linewidth=0.5)
    ax2.set_title(f'Cancelados por Unidade (Top 20)\n{periodo_str}', fontsize=14, fontweight='bold', pad=15)
    ax2.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax2.set_ylabel('', fontsize=11, fontweight='bold')
    ax2.set_xticks([])
    ax2.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
    ax2.invert_yaxis()
    xmax2 = top_cancelamentos_unidades.values.max() if len(top_cancelamentos_unidades) > 0 else 1
    ax2.set_xlim(0, xmax2 + max(3, int(0.15 * xmax2)))
    for i, v in enumerate(top_cancelamentos_unidades.values):
        ax2.text(v + 0.7, i, str(int(v)), va='center', fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'Sem dados disponíveis', ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title(f'Cancelados por Unidade (Top 20)\n{periodo_str}', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax2.set_ylabel('', fontsize=11, fontweight='bold')
    ax2.set_xticks([])

plt.tight_layout()

# Salvar a imagem SOMENTE com o nome referente ao dia final do período (ontem/yesterday ou domingo se segunda)
output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img"
os.makedirs(output_path, exist_ok=True)
nome_arquivo = f'graficos_unidades_{yesterday.strftime("%d-%m")}.png'
image_path = os.path.join(output_path, nome_arquivo)
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Gráficos salvos em: {image_path}')

plt.show()

C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_16708\2864555320.py:56: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax1.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_16708\2864555320.py:77: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax2.grid(axis='x', alpha=0.3, linestyle='--', visible=False)


Gráficos salvos em: C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img\graficos_unidades_18-02.png


C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_16708\2864555320.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### criando relação e imagem dos clientes cancelados

In [25]:
import matplotlib.patches as mpatches

if dia_semana == 0:  
    sexta = yesterday - datetime.timedelta(days=2)
    period_mask_cancel = df_cancelamentos["data_cancelamento"].isin([sexta, sexta + datetime.timedelta(days=1), yesterday])
    df_cancelamentos_periodo = df_cancelamentos[period_mask_cancel]
else:
    df_cancelamentos_periodo = df_cancelamentos[df_cancelamentos['data_cancelamento'] == yesterday]

# Agrupa e ordena por unidade e cliente, conta chassis
df_unidade_cliente = (
    df_cancelamentos_periodo
    .groupby(['unidade', 'cliente'])['chassi']
    .nunique()
    .reset_index()
)

# Seleciona as top 4 unidades com mais cancelamentos no total
total_por_unidade = (
    df_unidade_cliente
    .groupby('unidade')['chassi']
    .sum()
    .sort_values(ascending=False)
)
top_unidades = total_por_unidade.head(4).index.tolist()

# Filtra para manter apenas as top 4 unidades
df_top_unidade_cliente = df_unidade_cliente[df_unidade_cliente['unidade'].isin(top_unidades)]

# Ordena por unidade (top 4 ordem descendente do total), depois por chassi do cliente
df_top_unidade_cliente['unidade'] = pd.Categorical(df_top_unidade_cliente['unidade'], categories=top_unidades, ordered=True)
df_top_unidade_cliente = df_top_unidade_cliente.sort_values(['unidade', 'chassi'], ascending=[True, False])

print("Top 4 unidades com seus respectivos clientes e quantidades (cancelamentos):")
print(df_top_unidade_cliente)

# Gráfico de barras horizontais: cada barra é um cliente, cor codifica unidade, rótulo mostra a unidade + cliente + valor
import seaborn as sns

plt.figure(figsize=(14, 7))  # Aumenta o comprimento (largura) do gráfico horizontal
ax = plt.gca()

# Cria a coluna de label composta
df_top_unidade_cliente['label'] = df_top_unidade_cliente['unidade'].astype(str) + ' - ' + df_top_unidade_cliente['cliente'].astype(str)

# Ordenar para aparecer barras mais longas no topo
df_top_unidade_cliente = df_top_unidade_cliente.sort_values('chassi', ascending=True)

palette = sns.color_palette('tab10', n_colors=len(top_unidades))
unit_color = dict(zip(top_unidades, palette))

bars = ax.barh(
    df_top_unidade_cliente['label'],
    df_top_unidade_cliente['chassi'],
    color=[unit_color[un] for un in df_top_unidade_cliente['unidade']]
)

# Adiciona quantidade no fim de cada barra
for bar, val in zip(bars, df_top_unidade_cliente['chassi']):
    width = bar.get_width()
    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, str(int(val)), va='center', fontweight='bold')

# Legenda para cores das unidades
handles = [mpatches.Patch(color=unit_color[un], label=un) for un in top_unidades]
ax.legend(handles=handles, title="Unidade", bbox_to_anchor=(1.02, 1), loc='upper left')

# Ajusta o limite do eixo x para caber melhor as barras e os rótulos
max_value = df_top_unidade_cliente['chassi'].max() if not df_top_unidade_cliente.empty else 1
ax.set_xlim(0, max_value + max(5, int(0.18 * max_value)))

plt.xlabel('Quantidade de chassis cancelados')
plt.ylabel('')
plt.title("Top 4 Unidades - Cancelamentos por Cliente", fontsize=15, fontweight='bold')
plt.tight_layout()

# Salvar imagem no diretório especificado
output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img"
os.makedirs(output_path, exist_ok=True)
nome_arquivo = f'top4_cancelamentos_unidade_cliente_{yesterday.strftime("%d-%m")}.png'
image_path = os.path.join(output_path, nome_arquivo)
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Imagem salva em: {image_path}')

plt.show()

Top 4 unidades com seus respectivos clientes e quantidades (cancelamentos):
                                      unidade  \
24                             UNIDADE CUIABA   
21                             UNIDADE CUIABA   
23                             UNIDADE CUIABA   
20                             UNIDADE CUIABA   
22                             UNIDADE CUIABA   
25                             UNIDADE CUIABA   
42                            UNIDADE VILHENA   
41                            UNIDADE VILHENA   
40                            UNIDADE VILHENA   
9   MICRO A - R A SERVICOS & CONSULTORIA LTDA   
10  MICRO A - R A SERVICOS & CONSULTORIA LTDA   
35                              UNIDADE SINOP   
36                              UNIDADE SINOP   
33                              UNIDADE SINOP   
34                              UNIDADE SINOP   

                                    cliente  chassi  
24    TLT EQUIPAMENTOS E TRANSPORTES EIRELI       7  
21                 ANDRADE TRAN

C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_16708\3496186264.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_top_unidade_cliente['unidade'] = pd.Categorical(df_top_unidade_cliente['unidade'], categories=top_unidades, ordered=True)


Imagem salva em: C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img\top4_cancelamentos_unidade_cliente_18-02.png


C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_16708\3496186264.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### automação envio whatsapp

In [26]:
# time.sleep(2)
# pyautogui.hotkey('win', 'e')
# time.sleep(4)
# pyautogui.hotkey('ctrl', 'l') 
# time.sleep(1.5)
# pyautogui.typewrite(r'C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(2.5)
# pyautogui.typewrite(r'graficos')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'c')
# #whatsapp - primeiro arquivo
# time.sleep(1.5)
# pyautogui.press('win')
# time.sleep(1.5)
# pyautogui.typewrite('whatsapp')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(1.5)
# pyautogui.typewrite("RELAT")
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'v')
# time.sleep(2.5)
# pyautogui.press('enter')
# time.sleep(1.5)
######################################################whatsapp - segundo arquivo
# pyautogui.hotkey('alt', 'tab')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'a')
# time.sleep(2.5)
# pyautogui.typewrite(r'tabela')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'c')
# time.sleep(1.5)
# pyautogui.hotkey('alt', 'tab')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'v')
# time.sleep(2.5)
# pyautogui.press('enter')


### carregando resultados em excel

In [27]:
import os

# save workbook to the full file path
wb.save(template)

name_file = fr'RELATORIO_ATIVACOES_CANCELAMENTOS_{yesterday}.xlsx'

output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\reports"
os.makedirs(output_path, exist_ok=True)
output_full_path = os.path.join(output_path, name_file)

# save workbook to the full file path
shutil.copy(template, output_full_path)

name_file_2 = fr'RELATORIO_ATIVACOES_CANCELAMENTOS.xlsx'

path_sharepoint = r"C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel"
full_path_sharepoint = os.path.join(path_sharepoint, name_file_2)

# copy the saved file to the SharePoint folder (src, dst)
shutil.copy(output_full_path, full_path_sharepoint)

wb.close()